# Lab03I — Seleção, liquidez e verificações H/I

**Execute tudo em CPU e salve uma cópia do notebook e o ZIP no Drive. Este é o único notebook a executar nesta entrega.** Ele inclui H (exclusões de 2020 e diagnóstico da calibração) e I (reconstrução do adendo de 2015 e associação com liquidez). Não precisa executar H separadamente nem repetir F/G/E ou os 6.000 geradores.

Usa X e os onze ZIPs COTAHIST de 2015–2025 já existentes no Drive, com hashes. Não consulta novas cotações. Processa os dois universos e todos os blocos no mesmo lote. Há progresso por ano, universo e bloco.

Preserva os 80/117 principais, ativo/classe/ISIN, dez setores e ausências. O universo de 2015 é uma sensibilidade fixa. A reconstrução literal do avaliador é diagnóstica; a variante interpretada preserva as sete lacunas documentadas, incluindo os cinco dias de direitos. Os 83 eventos são marcações heurísticas, não homologação econômica. Liquidez é associação pós-parecer com inferência aproximada, sem identificação causal ou confirmação independente.


In [ ]:
import os,sys,json,hashlib,zipfile,subprocess,shutil,importlib.util
from pathlib import Path
from datetime import datetime,timezone
from urllib.request import urlretrieve
for module,package in [('numpy','numpy'),('pandas','pandas'),('scipy','scipy'),('matplotlib','matplotlib'),('threadpoolctl','threadpoolctl')]:
    if importlib.util.find_spec(module) is None:subprocess.run([sys.executable,'-m','pip','install','--quiet',package],check=True)
import numpy as np,pandas as pd
try:
    from IPython.display import display,Image
except ImportError:
    def display(value):print(str(value)[:6000])
    def Image(filename):
        from PIL import Image as PillowImage
        im=PillowImage.open(filename);im.load();return filename
LOCAL_MODE=bool(os.environ.get('ALEM_BETA_LOCAL'))
os.environ['ALEM_BETA_EXECUTION_ENV']='local_notebook_integration' if LOCAL_MODE else 'Colab'
BASE=Path(os.environ.get('ALEM_BETA_LOCAL','/content/alem_beta_lab03i')).resolve();BASE.mkdir(parents=True,exist_ok=True)
STAMP=datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN=BASE/('run_'+STAMP);RUN.mkdir();ROOT=RUN/'runtime';ROOT.mkdir()
def digest(p):return hashlib.sha256(Path(p).read_bytes()).hexdigest()
def script(name,*args):subprocess.run([sys.executable,str(ROOT/'scripts'/name),*map(str,args)],check=True)
print('Lab03I | CPU |', 'integração local, não Colab do autor' if LOCAL_MODE else 'Colab', '|',RUN.name)


## Fontes e cache

O código vem de um commit público imutável e cada arquivo tem SHA256. O cache X já existente é autenticado antes do cálculo. A pasta compartilhada `alem_do_beta/lab03f_cache` é reutilizada; se o arquivo não estiver lá, procura-se uma cópia idêntica no Drive antes de transferi-lo pelo ID conhecido. Não há nova coleta de mercado.


In [ ]:
SOURCE_COMMIT='4689c35c49a303e84f35b66c55320190fb630937'
SOURCE_PINS={'data/lab02/semantica_caixa_v1/recuperacao_fras_visivel_2024_2025.csv': '9888362d6022a1f92340ab204d534bc1a332df17146323a8945d363d2bef3682', 'data/lab03c/validacao_colab_20261023.json': '9c1f5c806b34d2153ae593b03c317f04f6e2058a6dafbe931713a06cba569878', 'data/lab03d/adendo_classes_equivalencia_v1.json': '019f36cb690f0b50e27e9a6be16fd1347ab55a1e4508803c160ba49c4aaa2ca5', 'data/lab03d/adendo_envelope_v1.json': 'd6d1df31e9fec714f44abeec66f61ad75117b17a09cc2374ac73e311b7f99777', 'data/lab03d/adendo_guard_causal_v1.json': '68d23baf9ba304bb54a3bc59639c68df72bc67bbcc52258f8dfa293c6902613f', 'data/lab03d/adendo_temporal_lfc_v1.json': 'd3f1a9dc3169d47b3f8f2840b50cccf1ae2de2033f5f48d95edca8a85a1073be', 'data/lab03d/inputs_manifest.json': '31225c9179356dc027a9fce2c86c4de533649ca3eac4016cf7d6832f8f5d7e5a', 'data/lab03d/plano_v1.json': 'f60b8422654df4cb27eccf69bacde50be23289cab4116a0396011ebf7daf0891', 'data/lab03d/validacao_colab_20261023.json': '6d7f60a1e56087e8a0ebbc329d3a4948028be768359433313d5b194252407fa7', 'data/lab03e/inputs_manifest.json': '052617b050e5da19a922d4c0c036f342219936dddeb1348d7d92ca4666b77a5e', 'data/lab03e/plano_v1.json': 'b8aa04548847dfa66b952e043d533233bfc8aa0c69350346cc9e145ccf3c0555', 'data/lab03e/primary/fras_cash.xlsx': '3597c6058a8358bd70c5a416924fe804cebc77feb659afaa34288053f6794df3', 'data/lab03e/primary/fras_cash_page.html': 'abfab5b634460f87c3688d23c571f77bad9a128932379aab37f4a120ca59f510', 'data/lab03e/primary/fras_extraction_audit.json': 'f8895cd3d04055417b1ad0a0fe7361621d5f1100075a56c04d1791eccb8e78da', 'data/lab03e/primary/fras_published_cash_inventory.csv': '90724375f4e8fa47fe91642186a80d48ba823db978920d72a435f3236696fbc3', 'data/lab03e/primary/retrieval.json': '3efd871a1073f92a734d831b302dbff05393f50e7a9567d850b057af29e8b3fc', 'data/lab03e/primary_sources_manifest.json': '81966a21513e79b3bd7e26edbf4e01d2ec778cdb916b24d1eb3aa66120b8c21f', 'data/lab03f/freeze.json': '7b69a88fac384f9a0d4515c0093407966eadec5abf848250c6eafa8bbe6d0388', 'data/lab03f/plano_pos_parecer_v1.json': 'f94cac62a6aa03249fb102ae7f74d5f45f3c28ca4968e503899f11b07cff6d1f', 'data/lab03g/inputs_manifest.json': '9f9fc53767bf3a95ab9fd562744cbb19466f8db65fbea7298cb7f63d230bfe15', 'data/lab03g/plano_v1.json': 'd347bfbbcef27e96aa8e8db392cb72b172a09d07140e4a3da90753fd76419da1', 'data/lab03g/validacao_colab_20261024.json': '7d005efaa2cd889f9af40b3b5797bccec962ec3c9725ca6698fe558e6dcdb94c', 'data/lab03h/inputs_manifest.json': '3b54eac63f7fb55d2c122939d2e2ac61566fa14785df0b267d2cf85b49c8d3b1', 'data/lab03h/plano_v1.json': 'c792d6b12948418ee54f904fa9fad75ea40156fc6236df881467f161638e6511', 'data/lab03i/adendo_associacoes_v1.json': '0a1e35def0bf5df7aa176bb105fe8e9be8798e76c6733201280d887541673dc5', 'data/lab03i/adendo_defasagens_v1.json': '5af2c2ae97241afe420b9f5c82803ed7e5b5659b70ae9a606a6e6320a7f9a709', 'data/lab03i/adendo_lacunas_v1.json': '5422cc22d0fbe98d88c5a21af68d4b53e8ed183fa84fcc3a6436bc6dac6b9659', 'data/lab03i/adendo_medida_liquidez.json': '3d1f9e9bb0cbf0790a54c5d42dad680f12154768ead1ced83a3da78a712bfdbf', 'data/lab03i/b3_cache_manifest.json': 'e9231565de3a81ea3a4586403ad55f50a5572442a69d4784546be2de28c675b8', 'data/lab03i/inputs_manifest.json': '1bdaf5cc016147ddad3640a8da4455b99ba8c0280d9992a4c02a607396ebb15c', 'data/lab03i/plano_v1.json': 'f12d34c8edc53d6bbb52abd8dfcbd6e69201efda1d49cb2cde9e9ecb59f308af', 'data/lab03i/recebimento_pacote.json': 'e1047e187350ebc0ed3e63af05fba79221e9e79c300df8b820cb316b0b97d9fa', 'data/lab03i/review_reference/eventos_detectados.csv': '4de560048a696c4d21df2ec061b90fb7f42b64a9f709b0767f314d8124c5bc90', 'data/lab03i/review_reference/market_returns_2015_2025.csv.gz': 'cdee511530b44db995fb8513c4931717c69498e458dd90495caa40261a4a8427', 'data/lab03i/review_reference/por_ativo_originais_COTAHIST.csv': '562c00c49aa950bff8459b3b78eb435e161ddf83c171962df499a2163b658a57', 'data/lab03i/review_reference/por_ativo_universo_2015.csv': '22c33f3db9815a05cc2c4f42a4da35fbaf3f94dabf9026dbbe849f9f11912916', 'data/lab03i/review_reference/resultados_agregados.csv': 'ac96d17780b4e9c21da5733fc25b5a62497b85c10216018e38c8d55386334f34', 'data/lab03i/review_reference/saidas_para_revisao.csv': '58de45a0bdf9962ef4f6ed022dbfb43a42a42f21af7a25f1de27e7c3e379ca38', 'data/lab03i/review_reference/universo_2015.csv': '319ba3c0faae6d9488b9a3fd58fbbbefd0cdd73c8492c826f888ad5536a9f595', 'data/lab03i/testes_locais.json': 'ffdfc848eaece24177b9f7f92fbafe629f160756ba849219d440242bdf6bd310', 'notebooks/Lab03D_inferencia_previsao.ipynb': '8ee5147c232efd43cdf1299d1059b5e8d43dc0a673b33381ab33e736b177a43f', 'notebooks/Lab03G_mudanca_calibracao.ipynb': '3ee7e9bc48c89e18bd81813d947d43b415c05f27a5f9ebc5b5c77c39c996da88', 'results/tables/lab03_colab_20261023/annual_primary.csv.gz': '7eae40bad9c9670fcc3954d72914b89a2731a68e204176e21e1e0483c8afde8b', 'results/tables/lab03b_colab_20261023/annual_sector.csv': '0252458058c8f23d56dbfa2e2ac8d0edd99e441271b02a1f8eb8a10b52003e37', 'results/tables/lab03b_colab_20261023/asset_betas.csv.gz': '0c2c54fc1f212d914cf083300757e2706a94d32ae762e64b8814e1a837192275', 'results/tables/lab03b_colab_20261023/risk_annual.csv.gz': '3d11280b802a72bbefe4405c78a5fd9254dd409f1d9c869cd4fde359d0909184', 'results/tables/lab03b_colab_20261023/sector_eta.csv': 'bfe546aa387ccf98ab4af40959e95b9f5b42f5697a88f7ce915c8cedeac3fee0', 'results/tables/lab03b_colab_20261023/sector_point.csv': '83cf15a21781921804feba92fa7788d424c378425c45c18493fba89dfefae0e2', 'results/tables/lab03e_colab_20261023/asset_nuisance.csv.gz': 'ba398bf2a7ba282f3dabbf7f63c41c5379a8105de2bffc91788f25f3dbd86040', 'results/tables/lab03f_pos_parecer_20261024/aggregate.csv.gz': 'e4f1a4ff05ffc17194e826cd412eb651fe159a3055845cdcc62d7e08569e001a', 'results/tables/lab03f_pos_parecer_20261024/bootstrap_targets.npz': 'c9447e814c9aafc732a029978379f14a837ce950dd3b6ca5e04c12d19e55c1c1', 'results/tables/lab03f_pos_parecer_20261024/individual.csv.gz': 'd1e89113e8d8850563dd14713d67cb8cf026bcc78e1b70f517a70190e7815151', 'results/tables/lab03f_pos_parecer_20261024/profile_summary.csv': '1560a809f904e4f6c49ac2e5af9767f5b883c350d5278fddfa3895592190f2c4', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_cases.csv': '35e7263855f1c8d522e5d96ea4de3264a82f368843f6d16ba46149423f55dfc6', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_families.csv': 'a980077971a21590ed6a04eacc2140bf1487390a793ab8134296c303fb8aeb86', 'results/tables/lab03g_pos_parecer_v3_20261024/calibration_replications.csv.gz': 'cb6537858edd89e4517c3bf41e1e08d0dbb410c08f10ecdc0adb1e33594f9a61', 'results/tables/lab03g_pos_parecer_v3_20261024/observed_aggregate.csv.gz': 'c1b448523e4c0e86fc1d0aba3f2dbebbb4defa2878d066c720a669ad4f3a7a53', 'results/tables/lab03g_pos_parecer_v3_20261024/observed_draws.npz': '6fadf6e088965808bc4fc39eb470bd96def884c3d0128597141ce37d304e3702', 'results/tables/lab03g_pos_parecer_v3_20261024/temporal_asset_points.csv': '84cf4fe6196ca1d2be8a955bde2a356c1c96f80fedc7b0198763c86948a135ff', 'results/tables/lab03h_pos_parecer_v4_20261024/annual_overview.csv': 'a0b40220f84edab0c56827aa2b30495131747cf31759875ac89c0d58041edebc', 'results/tables/lab03h_pos_parecer_v4_20261024/cdf.tex': 'd017d9c8de4de1802272a4ee45522f0c9a8e0894c184868541af9768dcc8c984', 'results/tables/lab03h_pos_parecer_v4_20261024/cdf_diagnostics.csv': '40b8b1bdf9b0db87a199ba09afd8c569b63cf967ea4abf6bc50d006ab795a265', 'results/tables/lab03h_pos_parecer_v4_20261024/coverage_by_target.csv': '72171fdc264cec7a65e0e7bc74909702a8f498ac557df63ad41e0d0db4411421', 'results/tables/lab03h_pos_parecer_v4_20261024/crisis.tex': 'a02fe03df8578d4ad05a67890c2dbf17379c3bdd78dbd83f3809d9a936e20af7', 'results/tables/lab03h_pos_parecer_v4_20261024/crisis_aggregate.csv.gz': '6c0a8389fdbd277f6c81a484516b38053df28fd16287231cd39bc63ca933d68e', 'results/tables/lab03h_pos_parecer_v4_20261024/crisis_asset_points.csv': '06e9ae646faca8185cee7468f007e6353a2b80118c4233dd3b2eeff9e6db5934', 'results/tables/lab03h_pos_parecer_v4_20261024/crisis_draws.npz': '50f81c79c29b655ab150b1ab12db443cd4f87786d5721a23611fc7e828809396', 'results/tables/lab03h_pos_parecer_v4_20261024/economic_scale.csv': '0f697da6a5216afa10f11e3b9f7aac142bb6c09f3cc613ab68f193e482d75686', 'results/tables/lab03h_pos_parecer_v4_20261024/execution.json': '82a015ad734d2d4dd9a88239541555f2dc78afb1ee87d1b4cfbf1d4f48178a6c', 'results/tables/lab03h_pos_parecer_v4_20261024/independent_verification.json': 'f22d120b1a082c250c461ba84e72aac99ff78d8d2754bc25f42099a58af4dea3', 'results/tables/lab03h_pos_parecer_v4_20261024/levels.tex': 'f96925b74c24e0d6759a62bb6cd7c40be6793496e41e4604dcb699da59b4e15c', 'results/tables/lab03h_pos_parecer_v4_20261024/microstructure.tex': 'b6b0cdf7d08f5ebbd8398a817770a2836e9096f714e64d1e25158ca3da3ea317', 'results/tables/lab03h_pos_parecer_v4_20261024/microstructure_raw_and_adjusted.csv': '19a7074fed9bc9ccc4d2dd1916b2b2c0dda8121bc40e9fdc0ffa055ad47e991c', 'results/tables/lab03h_pos_parecer_v4_20261024/normalization.csv': '47f8093f9d16e14cab790f9f72eef75622d32f67e1a1ecaf66c2aa6fac65dd97', 'results/tables/lab03h_pos_parecer_v4_20261024/period_observations.csv': '41bad75cb33d8850cfd0c9e46082da42c2dbcac2cc92f841492f1404f4878612', 'results/tables/lab03h_pos_parecer_v4_20261024/scale.tex': 'f704cddabefc5c2d32a2517647987baa47b4f7f2a1754b54fc951f74d0e66f4c', 'results/tables/lab03h_pos_parecer_v4_20261024/scale_diagnostics.csv': '5ae3fd79cd53d429abf33261209c311c7de070e69e3cfd6d87424daace63bea6', 'results/tables/lab03h_pos_parecer_v4_20261024/sector_summary.csv': '4f7f48ff2efd553247156a59a6ef75c07e4619c4ac38ada230213bf7e8ffd4f0', 'results/tables/lab03h_pos_parecer_v4_20261024/sectors.tex': '047f496b6ef4b442fbb4d5e5a17d14fd44af56c1cbc5548879e6170f9f232ef7', 'results/tables/lab03h_pos_parecer_v4_20261024/variance_concentration.csv': '19f647f133cfe3933055881624b54f8b369639ee38d1f4457e7b47d7620e237d', 'results/tables/lab03i_parecer_v4_20261024/aggregate_reproduced.csv': '8e80b89d98e462de4ffca6c3d2739e5b99f185c578ff32c3ee8a5d5e3eaaa811', 'results/tables/lab03i_parecer_v4_20261024/asset_liquidity.csv': 'bdd7d54335a661cddf59df909d04d41380bf4d31baa4d1ff4a00c8326bd3af16', 'results/tables/lab03i_parecer_v4_20261024/assets_originais_COTAHIST.csv': 'ceda8c680bdad9e200ed2055df32ed18016cd6b362f28404160dd99253fa5264', 'results/tables/lab03i_parecer_v4_20261024/assets_universo_2015.csv': '9de43cbfe5f61bc68ba2ebf508650ccc7bb792a186f7519aff26c6d3a84af1b3', 'results/tables/lab03i_parecer_v4_20261024/associations.tex': '740eb6419e3efc0f0e9cf98f704ddc390d3b3fbab34a911a1fd0965f9823cc13', 'results/tables/lab03i_parecer_v4_20261024/composition_groups.csv': '812652be2428a80b8781f27790fd4f910b73fb78c200e0fc74b7645bdc019789', 'results/tables/lab03i_parecer_v4_20261024/documented_gaps.csv': '89f9a09a44e70bf943d63a91c207caaa6253cd0a83c5935431a0773d360fdcfb', 'results/tables/lab03i_parecer_v4_20261024/events_reproduced.csv': '4de560048a696c4d21df2ec061b90fb7f42b64a9f709b0767f314d8124c5bc90', 'results/tables/lab03i_parecer_v4_20261024/gaps_aggregate_reproduced.csv': '2ac0350d06c6b191003e78ee97c6ef9aaddc62cc6c100e6762c16fa35bf57432', 'results/tables/lab03i_parecer_v4_20261024/gaps_assets_originais_COTAHIST.csv': '412c5511815d7e6209dd44e5eb655063841b5beaf66bba8f7c68798ccd14336e', 'results/tables/lab03i_parecer_v4_20261024/gaps_assets_universo_2015.csv': '82306ef7c15fd8fe00d073095569bf3d78abbe44107b8b03e3dd7879f5a4a8f0', 'results/tables/lab03i_parecer_v4_20261024/gaps_liquidity_joint.csv': '8322f03889f2491053bcb806850ed1f97febc9fe1203866842c0c6c5a40206d6', 'results/tables/lab03i_parecer_v4_20261024/independent_verification.json': 'a44951c1db4c055d089d33e3f901db4fe053e491cc18d6198090b3c26a69626a', 'results/tables/lab03i_parecer_v4_20261024/lag_asset_points.csv': '2c5100b6864d859b56b8d3f1dbb06e0e669dbc7451db0fc9c56e18bc148e6a6c', 'results/tables/lab03i_parecer_v4_20261024/lag_summary.csv': 'e848edc805bcaac31e3914031f084e81c1f347646699ba8ae71fcae9b91dc89b', 'results/tables/lab03i_parecer_v4_20261024/lag_verification.json': 'fafe080517e5ae4079c67002bf75e733e79c6e9a40203f3febf9ab774e9223cc', 'results/tables/lab03i_parecer_v4_20261024/lags.tex': 'f7132cb276b239a2343c0587d5724386ade45dc8cea0079276997fadbd808550', 'results/tables/lab03i_parecer_v4_20261024/liquidity_all_variants.csv': '595423f40c6e8c7e99efc49563f49b0b006c6b8f7708e1c249faf2539701356a', 'results/tables/lab03i_parecer_v4_20261024/liquidity_joint.csv': 'ae5d997ed8077adb1f959695739428f05600cc8b7885c744dc84e1734fd7251a', 'results/tables/lab03i_parecer_v4_20261024/replication.json': 'af2db8b7f8587c17170d6091b9ba3fc3a7394ccb7b8fef16f7c6ef6f9d2ad335', 'results/tables/lab03i_parecer_v4_20261024/selection.tex': '3b4575fdec4f6c87de5b4ebb28b74f1d54f21aec6ef72106173590b8db771e86', 'results/tables/resposta_pareceres_20261024/inventario_117.csv': '6344e9f908dbfe2563296d9ad89e5a74e8848dca23fe3cfbade439381d0b1982', 'scripts/build_lab03e_notebook.py': 'f7cc2dd6861ee77ef3221a22e6545ec25dcde1a6fbd7f0e4abac512a1e80d16d', 'scripts/build_lab03e_runtime.py': 'd8ff2d9de46d84b2fdaddfd26f9849a54ac2949dc28aa2676531eae330feb86a', 'scripts/export_lab03d.py': '413ec36ee2f63415a76489706a2f7de482410c81263677a3c71c61c2d598a701', 'scripts/export_lab03e.py': '55f88b6a8a83379c24daf9d7b42d0b49f2bb2219b2a61e46faa438131144e1f9', 'scripts/extract_lab03e_fras.py': '7107ebcedb172a1a734ef0b36a4a4bae207bc815c9bf0c45d27f422c9e81175d', 'scripts/plot_lab03d.py': 'd7d6c63adabf7ff963bec34e7b65a57fc8bc34523a13c8c0ead70bfe865ad6ec', 'scripts/plot_lab03e.py': 'db0dac0930bb8bcc1bb3e3497bd6db1be95615c8a8bd0471cd813878bfae8f43', 'scripts/plot_lab03g.py': '61b12abf72d2f7e7ebf55bafe4fb8a505b9812fdee432dad50f5265905c6b697', 'scripts/plot_lab03h.py': '6f0f0e7dce7ffb62644a6c32eea804d62cffe9289dd6a88b1f90dad55e14f827', 'scripts/plot_review_2015.py': '3c6ba81da6d284b0f30987cd8e7eb48e99b79a7791c1003267401839d4b88bdf', 'scripts/replicate_review_2015.py': 'a337500b857ef5fbb09c939a1515378175b4d4a22a54b042feef0ae48dbce4c7', 'scripts/run_lab03d.py': 'aa689b2b6ff203b77daca85b09923804a719c0d08cc6b4a29b5e95a1145b9975', 'scripts/run_lab03d_lfc.py': '5110d5db922e1f1ad46867aeb5d123aa9af58bf97d9dc7d04341d2704d770151', 'scripts/run_lab03d_rank.py': '3e100649f26e9c21e5ed7dd9426fce4dd3c2184d735db2e96c6126d8fb06cd4d', 'scripts/run_lab03e_economic.py': '3ec83455d75523d854813afc908bc59b6658c1d9bd984350c13abdf1982b075a', 'scripts/run_lab03e_inference.py': 'ff3c9aa2f870913330592ba85c1e1dfca974bae3f15e52fb7dc671c1033b2045', 'scripts/run_lab03e_prediction.py': '8aa01a0df2d2e9061cd33ac7b4990551b9648f33d0fcbe392a116701f1b5018f', 'scripts/run_lab03g.py': '44251bc1b8f248b2c7a471d466df2f7529e60e9ee952dcccf8d0608b1c9094f1', 'scripts/run_lab03h.py': '7600786be41e089f89a5b49f02bbef018f81ce598b8ffac346a5f17e07695bb7', 'scripts/sensitivity_review_2015_lags.py': '2fc6493e7ee076a20f25e071fac21ae5fa95cfce5a6a38ef851bcb8ba6787ad1', 'scripts/validate_lab03d_diagnostics.py': '659bee564d506b2118d9a388a6b48b1d8461716dafdb127c099ba9a443df0f32', 'scripts/validate_lab03d_outputs.py': 'd21eb2f7a57f993b9b1a9ea38b8ebe42cdcd3f075b5fc6fd185bf449d5f96d26', 'scripts/validate_lab03d_rank.py': 'b4c178dc13b51dd72c3e4174255f58b7bb453f9a1ccced1284bb615754f559d4', 'scripts/verify_lab03e_economic.py': '0c51cf44ab70e2bdcacb6fe215b1eebbce2d6a6e290796cf2c1244f4a481fa34', 'scripts/verify_lab03e_inference.py': '211f9b9be0aa693bf1ae3c965e02e054700051daed3f7b694e14482ca002ac4d', 'scripts/verify_lab03e_prediction.py': 'ef39b6b3af27a9bd11d23ae60e3a5b0b54a857f05cf5cccb351452272d686ad2', 'scripts/verify_lab03g.py': '6ee9d1f04570f5292c920f8d67c17d71cf6a728a2d165019a4a973b135d8595b', 'scripts/verify_lab03h.py': 'e20bf9da087f63c75c562c8c653bcb2ccfa1b56ed788f72c74c309dda0d95923', 'scripts/verify_review_2015.py': '1b87eaac11ce21ae1439e859c9cc276094ec4d08657b365b7ba996be2ee9aadd', 'src/lab02.py': 'f83bef7d5da251fb2571699572f202ee16a8b007522f3e5f1f6f1385f8540b0f', 'src/lab03c.py': '715b9cd9cec7d41a54d129fde497c14a68013a66e80a337317d91c8a2c4b20dd', 'src/lab03d_envelope.py': 'c24eab65913eb9dc3ccd76c7db962820aedc6239cb75f696c80f6755bc53db4a', 'src/lab03d_inference.py': '0d7a891a94c98539168a382bbab2ae7f16aeebf338398a2c6e98ef9498077708', 'src/lab03d_prediction.py': 'e52cad9acfefce1c27f8f8b9e3d479b0d942cd036946173ef7381fa39c05088e', 'src/lab03d_rank.py': '815799908d43bf3b0bc2c71d01750d619ae8522e050b82e426d4aee9239faa53', 'src/lab03e_common.py': '179dc3a15289f83d4780cbed88dfd07909e813f5511f87531626626419be0d96', 'src/lab03e_economic.py': '196875220ee4a64f56d023ed1df121a8cb5d0089788cfd4a64f658d6ec49dafd', 'src/lab03e_inference.py': '73ee08c05a1e0fa14cdba9f591db90e99ae877050d2dfd8622c5bf3fa7c0e517', 'src/lab03e_prediction.py': 'b6010fd76bf51c822635a994c787994c9c5f8221236d2137975096ed18fbec06', 'src/lab03f.py': 'e65dc527403d5a444b7b52caa5bfd0bde341a48a7b3b57979990c422478c68b7', 'src/lab03g.py': 'e89d3723c71da6accecd469209ce39d1e902cd0e620b83a8ac28103ac11e8c9e', 'src/lab03h.py': '3c999a1b4f9188d1dc80121fd34ebbedab2d9225b46bf03912260893d0c980c7', 'tests/test_lab03d.py': '3639cb28cf554812a10c8d4e6ca04f6baf298be19ba5a6521d30827bb03b8b92', 'tests/test_lab03e.py': 'f72b6901926edd80c8815b79b32c6daedd9210c062db199b7232b20d01c72d5f', 'tests/test_lab03g.py': 'eb402449b1ebed3eb01acf8184439973bff1d00ea8ce149d060d8e58a03e613c', 'tests/test_lab03h.py': 'd025121b0b3ea9c33bcfe86a9abdc2a3756875322e5efa47f83feb00b91c425f', 'tests/test_review_2015.py': '6ed2f509e9c4d09d5671b778416d69a3dbf1523e91c9d09e326248dd3afafe57'}
for name,expected in SOURCE_PINS.items():
    target=ROOT/name;target.parent.mkdir(parents=True,exist_ok=True)
    if LOCAL_MODE:shutil.copy2(Path(os.environ['ALEM_BETA_LOCAL_SOURCE'])/name,target)
    else:urlretrieve('https://raw.githubusercontent.com/robsondf/alem-do-beta-gemini/'+SOURCE_COMMIT+'/'+name,target)
    assert digest(target)==expected,'Fonte divergente: '+name
print('Fontes autenticadas:',len(SOURCE_PINS),'| commit',SOURCE_COMMIT)
X_ID='1aExT7v-GD8s5xKzM0K-0LIIJER8E8Twg'
X_SHA='297da2afde534082c398324f308df7af134c9692e7d61d6e0b2b820341ea5214'
if LOCAL_MODE:
    X_ARCHIVE=Path(os.environ['ALEM_BETA_LOCAL_X'])
else:
    from google.colab import auth,drive
    auth.authenticate_user();drive.mount('/content/drive')
    import google.auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    credentials,_=google.auth.default();service=build('drive','v3',credentials=credentials,cache_discovery=False)
    CACHE=Path('/content/drive/MyDrive/alem_do_beta/lab03f_cache');CACHE.mkdir(parents=True,exist_ok=True)
    X_ARCHIVE=CACHE/(X_ID+'.zip')
    if not X_ARCHIVE.exists() or digest(X_ARCHIVE)!=X_SHA:
        name=service.files().get(fileId=X_ID,fields='name').execute()['name']
        found=None
        for candidate in Path('/content/drive/MyDrive').rglob(name):
            if candidate.is_file() and digest(candidate)==X_SHA:found=candidate;break
        if found is not None:X_ARCHIVE=found
        else:
            part=X_ARCHIVE.with_suffix('.part')
            with part.open('wb') as stream:
                download=MediaIoBaseDownload(stream,service.files().get_media(fileId=X_ID));done=False
                while not done:_,done=download.next_chunk()
            assert digest(part)==X_SHA;part.replace(X_ARCHIVE)
assert digest(X_ARCHIVE)==X_SHA
print('Cache X autenticado:',X_ARCHIVE.name,'|',X_ARCHIVE.stat().st_size,'bytes')

official=json.loads((ROOT/'data/lab03i/b3_cache_manifest.json').read_text())
if LOCAL_MODE:B3_CACHE=Path(os.environ['ALEM_BETA_LOCAL_B3_CACHE'])
else:
    B3_CACHE=RUN/'b3_cache';B3_CACHE.mkdir()
    original=Path('/content/drive/MyDrive/Além do Beta/cache_b3')
    if not original.is_dir():
        candidates=Path('/content/drive/MyDrive').rglob('COTAHIST_A2015.ZIP')
        first=next(candidates,None)
        if first is not None:original=first.parent
    for item in sorted(official,key=lambda x:x['name']):
        name=item['name'];target=B3_CACHE/name;candidate=original/name
        if candidate.is_file() and digest(candidate)==item['sha256']:target.symlink_to(candidate)
        else:
            with target.open('wb') as stream:
                download=MediaIoBaseDownload(stream,service.files().get_media(fileId=item['drive_id']));done=False
                while not done:_,done=download.next_chunk()
        assert digest(target)==item['sha256'],name
        print('Cache oficial autenticado:',name,'|',item['bytes'],'bytes')
    shutil.copy2(ROOT/'data/lab03i/b3_cache_manifest.json',B3_CACHE/'authenticated_manifest.json')
for item in official:assert digest(B3_CACHE/item['name'])==item['sha256'],item['name']
print('Onze caches B3 autenticados; nenhuma nova cotação consultada.')


## Métodos e limites

H: exclusões somente da metade inicial, Holm4 novo por bloco e famílias anteriores preservadas. I: seleção única em 2015, perdas de elegibilidade explícitas, reprodução literal das 83 marcações e variante com as sete lacunas preservadas. A reamostragem é conjunta por sessão, sem tratar ativos como observações iid.

As associações de postos foram declaradas após o parecer e o resultado externo inicial; Holm8 inclui quatro associações em duas variantes, por bloco. Mantêm-se 21 como principal e 5/63 como sensibilidades. O deslocamento bootstrap é reportado; não houve calibração específica de tamanho/cobertura. Não selecionar variante, universo ou bloco pela significância.


In [ ]:
for name in ['test_lab03h.py','test_review_2015.py']:
    subprocess.run([sys.executable,'-m','unittest','discover','-s',str(ROOT/'tests'),'-p',name,'-v'],check=True)
OUTPUT=RUN/'outputs';I_OUTPUT=RUN/'outputs_I'
script('run_lab03h.py','--x-archive',X_ARCHIVE,'--output',OUTPUT)
script('replicate_review_2015.py','--b3-cache',B3_CACHE,'--review-dir',ROOT/'data/lab03i/review_reference','--x-archive',X_ARCHIVE,'--output',I_OUTPUT)


In [ ]:
script('verify_lab03h.py','--x-archive',X_ARCHIVE,'--output',OUTPUT)
script('plot_lab03h.py','--output',OUTPUT)
reference=ROOT/'results/tables/lab03h_pos_parecer_v4_20261024'
compared=0;threshold_positions=0
for expected_path in sorted(reference.glob('*.csv*')):
    name=expected_path.name
    observed=pd.read_csv(OUTPUT/name,float_precision='round_trip');expected=pd.read_csv(expected_path,float_precision='round_trip')
    pd.testing.assert_frame_equal(observed,expected,check_exact=False,atol=1e-10,rtol=1e-9);compared+=observed.size
    for col in observed.select_dtypes(include='number'):
        if col.startswith('p_'):
            np.testing.assert_array_equal(observed[col].le(.05),expected[col].le(.05));threshold_positions+=len(observed)
with np.load(OUTPUT/'crisis_draws.npz') as actual,np.load(reference/'crisis_draws.npz') as expected:
    assert actual.files==expected.files
    for key in actual.files:np.testing.assert_allclose(actual[key],expected[key],atol=1e-10,rtol=1e-9);compared+=actual[key].size
for name,expected in SOURCE_PINS.items():assert digest(ROOT/name)==expected,name
record={'passed':True,'environment':'local_notebook_integration' if LOCAL_MODE else 'Colab','source_commit':SOURCE_COMMIT,'source_files':len(SOURCE_PINS),'compared_values':compared,'p_threshold_positions':threshold_positions,'atol':1e-10,'rtol':1e-9,'X_sha256':digest(X_ARCHIVE),'python':sys.version,'numpy':np.__version__,'pandas':pd.__version__,'new_outer_generators':0,'external_2015_package_reproduced':False}
(OUTPUT/'notebook_comparison.json').write_text(json.dumps(record,indent=2)+'\n')
print(json.dumps(record,indent=2))
script('plot_review_2015.py','--output',I_OUTPUT)
script('sensitivity_review_2015_lags.py','--output',I_OUTPUT)
script('verify_review_2015.py','--output',I_OUTPUT)
iref=ROOT/'results/tables/lab03i_parecer_v4_20261024';iv=0;it=0
for path in sorted(iref.glob('*.csv')):
    name=path.name
    actual_path=I_OUTPUT/'gaps_preserved'/name.removeprefix('gaps_') if name.startswith('gaps_') else I_OUTPUT/name
    actual=pd.read_csv(actual_path,float_precision='round_trip');expected=pd.read_csv(path,float_precision='round_trip')
    np.testing.assert_array_equal(sorted(actual.columns),sorted(expected.columns))
    pd.testing.assert_frame_equal(actual.reindex(columns=expected.columns),expected,check_exact=False,atol=1e-10,rtol=1e-9);iv+=actual.size
    for col in actual.select_dtypes(include='number'):
        if col=='p' or col.startswith('p_'):
            np.testing.assert_array_equal(actual[col].le(.05),expected[col].le(.05));it+=len(actual)
for name,expected in SOURCE_PINS.items():assert digest(ROOT/name)==expected,name
record.update(stage='H_and_I',external_2015_package_reproduced=True,run_stamp_UTC=STAMP,I_compared_values=iv,I_p_threshold_positions=it,I_independent_verification=json.loads((I_OUTPUT/'independent_verification.json').read_text()),only_existing_caches=True)
(I_OUTPUT/'notebook_comparison.json').write_text(json.dumps(record,indent=2)+'\n')
print(json.dumps(record,indent=2))


In [ ]:
display(pd.read_csv(OUTPUT/'crisis_aggregate.csv.gz').query('block==21')[['profile','statistic','estimate','p_centered','p_holm4']])
display(pd.read_csv(I_OUTPUT/'gaps_preserved/aggregate_reproduced.csv').query('bloco==21'))
display(pd.read_csv(I_OUTPUT/'liquidity_all_variants.csv').query('variant=="gaps_preserved" and block==21'))
for directory,name in [(OUTPUT,'lab03h_crise'),(OUTPUT,'lab03h_periodo_completo'),(I_OUTPUT,'lab03i_liquidez')]:display(Image(filename=str(directory/'figures'/(name+'.png'))))


## Interpretação e salvamento

As comparações usam retornos; preços são insumos de reconstrução. Nenhuma ponte vira retorno diário. A assimetria e a mudança dependem do universo e da elegibilidade. Os dados são compatíveis com uma relação de liquidez, sem identificar causa, efeito de melhora de liquidez, retorno de carteira ou prêmio de risco.

Os ICs são nominais, não uniformemente validados. A calibração anterior não se transfere às novas máscaras, universos ou correlações. A reamostragem conjunta preserva dependência, mas há deslocamento bootstrap e seleção retrospectiva. A auditoria econômica global e KRR permanecem incompletos.

O ZIP contém fontes, tabelas, gráficos, bancos bootstrap e matrizes derivadas de retorno para auditoria. Os ZIPs brutos COTAHIST não são incluídos. **Mantenha o export no Drive; as matrizes não integram o repositório público. Salve também este notebook executado e envie o link da cópia.**


In [ ]:
export=RUN/('resultados_lab03i_'+STAMP+'.zip')
inventory={}
with zipfile.ZipFile(export,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for base,prefix in [(ROOT,'runtime'),(OUTPUT,'outputs'),(I_OUTPUT,'outputs_I')]:
        for path in sorted(base.rglob('*')):
            if path.is_file() and '__pycache__' not in path.parts:
                name=prefix+'/'+str(path.relative_to(base));z.write(path,name);inventory[name]=digest(path)
    z.writestr('manifesto_exports.json',json.dumps(inventory,indent=2))
with export.open('rb') as stream:os.fsync(stream.fileno())
with zipfile.ZipFile(export) as verify_zip:
    assert verify_zip.testzip() is None
    saved=json.loads(verify_zip.read('manifesto_exports.json'))
    assert set(saved)==set(verify_zip.namelist())-{'manifesto_exports.json'}
    for name,expected in saved.items():assert hashlib.sha256(verify_zip.read(name)).hexdigest()==expected,name
print('Export:',export.name,'| bytes:',export.stat().st_size,'| SHA256:',digest(export))
if not LOCAL_MODE:
    folder=Path('/content/drive/MyDrive/alem_do_beta/lab03i_resultados');folder.mkdir(parents=True,exist_ok=True)
    destination=folder/export.name;shutil.copy2(export,destination);assert digest(destination)==digest(export)
    print('SALVO NO DRIVE:',destination)
else:print('Integração local concluída. Não é execução Colab do autor.')
